# MS2

 Initial Data Preprocessing and Triplet Generation (pre-canonicalization)

In [ ]:
# --- Installations (if running in a fresh environment) ---
!pip install neo4j

import pandas as pd
import spacy
from neo4j import GraphDatabase
from datasets import load_dataset
import re
from tqdm import tqdm

# 1. Load Enron dataset
dataset = load_dataset("corbt/enron-emails", split="train")

# 2. Convert to DataFrame
df = pd.DataFrame(dataset)

# 3. Clean and preprocess df
# a. Selecting and renaming columns
df = df[["from", "to", "subject", "body"]]
df.columns = ["from_email", "to_email", "subject", "body"]

# b. Dropping NaNs
df = df.dropna(subset=["from_email", "to_email", "subject", "body"])

# c. Ensuring 'to_email' is a single string
df['to_email'] = df['to_email'].apply(lambda x: x[0] if isinstance(x, list) else x)

# d. Dropping duplicate rows
df = df.drop_duplicates()

# e. Resetting the DataFrame index
df = df.reset_index(drop=True)

# f. Defining and applying the clean_text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\\s+', ' ', text)
    text = re.sub(r'\\S*@\\S*', '', text) # Remove email addresses
    text = re.sub(r'[^\\w\\s@.]', '', text)
    return text.strip()

df["clean_subject"] = df["subject"].apply(clean_text)
df["clean_body"] = df["body"].apply(clean_text)

# g. Concatenating into 'full_text'
df["full_text"] = df["clean_subject"] + " " + df["clean_body"]

# h. Extracting sender and receiver domains
df["sender_domain"] = df["from_email"].apply(lambda x: x.split("@")[-1])
df["receiver_domain"] = df["to_email"].apply(lambda x: x.split("@")[-1])

# i. Converting to lowercase
df["from_email"] = df["from_email"].str.lower()
df["to_email"] = df["to_email"].str.lower()

# 4. Load the 'en_core_web_sm' spaCy model
nlp = spacy.load("en_core_web_sm")

# 5. Define the extract_entities function
def extract_entities(text, min_len=2):
    doc = nlp(text)
    persons = []
    orgs = []
    locations = []
    dates = []

    for ent in doc.ents:
        if len(ent.text.strip()) > min_len:
            if ent.label_ == "PERSON":
                persons.append(ent.text)
            elif ent.label_ == "ORG":
                orgs.append(ent.text)
            elif ent.label_ in ["GPE", "LOC"]:
                locations.append(ent.text)
            elif ent.label_ == "DATE":
                dates.append(ent.text)

    return persons, orgs, locations, dates

# 6. Create a sample DataFrame df_sample
df_sample = df.head(200).copy()

# 7. Apply the extract_entities function
df_sample["persons"], df_sample["orgs"], df_sample["locations"], df_sample["dates"] = zip(
    *df_sample["full_text"].apply(extract_entities)
)

print("df_sample head after entity extraction with email removal:")
print(df_sample.head())

# --- Triplet Generation Logic ---
def combine_entities(row):
    combined = []
    for p in row['persons']:
        combined.append((p, 'PERSON'))
    for o in row['orgs']:
        combined.append((o, 'ORG'))
    for l in row['locations']:
        combined.append((l, 'LOCATION'))
    for d in row['dates']:
        combined.append((d, 'DATE'))
    return combined

df_sample['all_entities'] = df_sample.apply(combine_entities, axis=1)

# Generate 'RELATED_TO' triplets from all pairs of entities within each email
triplets_related_to = []

for index, row in df_sample.iterrows():
    entities = row["all_entities"]

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            triplets_related_to.append({
                "subject": entities[i][0],
                "subject_type": entities[i][1],
                "relation": "RELATED_TO",
                "object": entities[j][0],
                "object_type": entities[j][1]
            })

triplet_related_to_df = pd.DataFrame(triplets_related_to)
print("Generated co-occurrence triplets from df_sample (after email removal):")
print(triplet_related_to_df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 30.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/581 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/167M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/517401 [00:00<?, ? examples/s]

df_sample head after entity extraction with email removal:
                from_email                 to_email    subject  \
0  phillip.allen@enron.com     tim.belden@enron.com              
1  phillip.allen@enron.com  john.lavorato@enron.com        Re:   
2  phillip.allen@enron.com   leah.arsdall@enron.com   Re: test   
3  phillip.allen@enron.com    randall.gay@enron.com              
4  phillip.allen@enron.com     greg.piper@enron.com  Re: Hello   

                                                body clean_subject  \
0                          Here is our forecast\n\n                  
1  Traveling to have a business meeting takes the...                 
2                     test successful.  way to go!!!             s   
3  Randy,\n\n Can you send me a schedule of the s...                 
4                Let's shoot for Tuesday at 11:45.                   

                                          clean_body  \
0                                                 ss   
1  ssss.ss.

Refined Entity Extraction and Triplet Generation

In [ ]:
import pandas as pd
import spacy
from neo4j import GraphDatabase
from datasets import load_dataset
import re
from tqdm import tqdm

# 1. Load Enron dataset
dataset = load_dataset("corbt/enron-emails", split="train")

# 2. Convert to DataFrame
df = pd.DataFrame(dataset)

# 3. Clean and preprocess df
# a. Selecting and renaming columns
df = df[["from", "to", "subject", "body"]]
df.columns = ["from_email", "to_email", "subject", "body"]

# b. Dropping NaNs
df = df.dropna(subset=["from_email", "to_email", "subject", "body"])

# c. Ensuring 'to_email' is a single string
df['to_email'] = df['to_email'].apply(lambda x: x[0] if isinstance(x, list) else x)

# d. Dropping duplicate rows
df = df.drop_duplicates()

# e. Resetting the DataFrame index
df = df.reset_index(drop=True)

# f. Defining and applying the clean_text function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\S*@\S*', '', text) # Remove email addresses
    text = re.sub(r'[^\w\s@.]', '', text)
    return text.strip()

df["clean_subject"] = df["subject"].apply(clean_text)
df["clean_body"] = df["body"].apply(clean_text)

# g. Concatenating into 'full_text'
df["full_text"] = df["clean_subject"] + " " + df["clean_body"]

# h. Extracting sender and receiver domains
df["sender_domain"] = df["from_email"].apply(lambda x: x.split("@")[-1])
df["receiver_domain"] = df["to_email"].apply(lambda x: x.split("@")[-1])

# i. Converting to lowercase
df["from_email"] = df["from_email"].str.lower()
df["to_email"] = df["to_email"].str.lower()

# 4. Load the 'en_core_web_sm' spaCy model
nlp = spacy.load("en_core_web_sm")

# 5. Define the extract_entities function
def extract_entities(text, min_len=2, min_len_person=3):
    doc = nlp(text)
    persons = []
    orgs = []
    locations = []
    dates = []

    for ent in doc.ents:
        current_min_len = min_len_person if ent.label_ == "PERSON" else min_len
        cleaned_ent_text = ent.text.strip().lower()

        # Post-processing rules: filter out specific noisy entities
        if cleaned_ent_text in ['enron', 'hou', 'ect']:
            continue

        if len(cleaned_ent_text) > current_min_len:
            if ent.label_ == "PERSON":
                persons.append(ent.text)
            elif ent.label_ == "ORG":
                orgs.append(ent.text)
            elif ent.label_ in ["GPE", "LOC"]:
                locations.append(ent.text)
            elif ent.label_ == "DATE":
                dates.append(ent.text)

    return persons, orgs, locations, dates

# 6. Create a sample DataFrame df_sample
df_sample = df.head(200).copy()

# 7. Apply the extract_entities function
df_sample["persons"], df_sample["orgs"], df_sample["locations"], df_sample["dates"] = zip(
    *df_sample["full_text"].apply(extract_entities)
)

print("df_sample head after entity extraction with email removal and refined filtering:")
print(df_sample.head())

# --- Triplet Generation Logic ---
def combine_entities(row):
    combined = []
    for p in row['persons']:
        combined.append((p, 'PERSON'))
    for o in row['orgs']:
        combined.append((o, 'ORG'))
    for l in row['locations']:
        combined.append((l, 'LOCATION'))
    for d in row['dates']:
        combined.append((d, 'DATE'))
    return combined

df_sample['all_entities'] = df_sample.apply(combine_entities, axis=1)

# Generate 'RELATED_TO' triplets from all pairs of entities within each email
triplets_related_to = []

for index, row in df_sample.iterrows():
    entities = row["all_entities"]

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            triplets_related_to.append({
                "subject": entities[i][0],
                "subject_type": entities[i][1],
                "relation": "RELATED_TO",
                "object": entities[j][0],
                "object_type": entities[j][1]
            })

triplet_related_to_df = pd.DataFrame(triplets_related_to)
print("Generated co-occurrence triplets from df_sample (after email removal and refined filtering):")
print(triplet_related_to_df.head())


df_sample head after entity extraction with email removal and refined filtering:
                from_email                 to_email    subject  \
0  phillip.allen@enron.com     tim.belden@enron.com              
1  phillip.allen@enron.com  john.lavorato@enron.com        Re:   
2  phillip.allen@enron.com   leah.arsdall@enron.com   Re: test   
3  phillip.allen@enron.com    randall.gay@enron.com              
4  phillip.allen@enron.com     greg.piper@enron.com  Re: Hello   

                                                body clean_subject  \
0                          Here is our forecast\n\n                  
1  Traveling to have a business meeting takes the...            re   
2                     test successful.  way to go!!!       re test   
3  Randy,\n\n Can you send me a schedule of the s...                 
4                Let's shoot for Tuesday at 11:45.        re hello   

                                          clean_body  \
0                               here is our f

Implement Entity Canonicalization

In [ ]:
!pip install fuzzywuzzy

Define and Apply canonicalize_entities Function

In [ ]:
from fuzzywuzzy import process

def canonicalize_entities(entity_list, threshold=80):
    if not entity_list: # Handle empty lists
        return []

    canonical_forms = []
    processed_entities = set() # To keep track of entities already canonicalized or absorbed

    # Sort entities by length in descending order to prioritize longer forms as canonical
    sorted_entities = sorted(list(set(entity_list)), key=len, reverse=True)

    for current_entity in sorted_entities:
        if current_entity in processed_entities:
            continue

        # Assume current_entity is a canonical form until a better match is found
        best_match = current_entity
        variations = [current_entity]

        for other_entity in sorted_entities:
            if current_entity == other_entity or other_entity in processed_entities:
                continue

            # Check for substring relationship or fuzzy match
            # Simple substring check: if one contains the other
            if current_entity in other_entity:
                # If a longer entity contains the current one, the longer one is potentially more canonical
                # For now, let's keep the `best_match` which is the current longer one being iterated
                variations.append(other_entity)
                processed_entities.add(other_entity)
            elif other_entity in current_entity:
                # Current entity is longer and contains the other, so current_entity is canonical
                variations.append(other_entity)
                processed_entities.add(other_entity)

        canonical_forms.append(best_match)
        processed_entities.add(best_match)

    return sorted(list(set(canonical_forms)))

# Apply canonicalization to df_sample
df_sample['persons'] = df_sample['persons'].apply(canonicalize_entities)
df_sample['orgs'] = df_sample['orgs'].apply(canonicalize_entities)
df_sample['locations'] = df_sample['locations'].apply(canonicalize_entities)
df_sample['dates'] = df_sample['dates'].apply(canonicalize_entities)

print("df_sample head after entity canonicalization:")
print(df_sample.head())
print("\nFirst few canonicalized persons:")
print(df_sample['persons'].head().to_string())

df_sample head after entity canonicalization:
                from_email                 to_email    subject  \
0  phillip.allen@enron.com     tim.belden@enron.com              
1  phillip.allen@enron.com  john.lavorato@enron.com        Re:   
2  phillip.allen@enron.com   leah.arsdall@enron.com   Re: test   
3  phillip.allen@enron.com    randall.gay@enron.com              
4  phillip.allen@enron.com     greg.piper@enron.com  Re: Hello   

                                                body clean_subject  \
0                          Here is our forecast\n\n                  
1  Traveling to have a business meeting takes the...            re   
2                     test successful.  way to go!!!       re test   
3  Randy,\n\n Can you send me a schedule of the s...                 
4                Let's shoot for Tuesday at 11:45.        re hello   

                                          clean_body  \
0                               here is our forecast   
1  traveling to have a b

Re-combine Entities and Regenerate Triplets

In [ ]:
def combine_entities(row):
    combined = []
    for p in row['persons']:
        combined.append((p, 'PERSON'))
    for o in row['orgs']:
        combined.append((o, 'ORG'))
    for l in row['locations']:
        combined.append((l, 'LOCATION'))
    for d in row['dates']:
        combined.append((d, 'DATE'))
    return combined

df_sample['all_entities'] = df_sample.apply(combine_entities, axis=1)

# Generate 'RELATED_TO' triplets from all pairs of entities within each email
triplets_related_to = []

for index, row in df_sample.iterrows():
    entities = row["all_entities"]

    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            triplets_related_to.append({
                "subject": entities[i][0],
                "subject_type": entities[i][1],
                "relation": "RELATED_TO",
                "object": entities[j][0],
                "object_type": entities[j][1]
            })

triplet_related_to_df = pd.DataFrame(triplets_related_to)
print("Generated co-occurrence triplets from df_sample (after canonicalization):")
print(triplet_related_to_df.head())

Generated co-occurrence triplets from df_sample (after canonicalization):
         subject subject_type    relation                 object object_type
0           1145         DATE  RELATED_TO                tuesday        DATE
1   next tuesday         DATE  RELATED_TO               thursday        DATE
2  john lavorato       PERSON  RELATED_TO                  keith      PERSON
3  john lavorato       PERSON  RELATED_TO           mike grigsby      PERSON
4  john lavorato       PERSON  RELATED_TO  monique sanchez frank      PERSON


Redefine batch_insert_triplets Function

In [ ]:
from neo4j import GraphDatabase

def batch_insert_triplets(tx, triplets_batch, subject_label_str, object_label_str):
    # Ensure labels are valid Cypher identifiers
    clean_subject_label = ''.join(e for e in subject_label_str if e.isalnum())
    clean_object_label = ''.join(e for e in object_label_str if e.isalnum())

    query = f"""
    UNWIND $triplets_batch AS t
    MERGE (s:{clean_subject_label} {{name: t.subject}})
    MERGE (o:{clean_object_label} {{name: t.object}})
    MERGE (s)-[:RELATED_TO]->(o)
    """
    tx.run(query, triplets_batch=triplets_batch)

print("The batch_insert_triplets function has been re-defined.")

The batch_insert_triplets function has been re-defined.


Neo4j Data Clearing, Index Creation, and Batch Insertion

In [ ]:
from neo4j import GraphDatabase
from tqdm import tqdm
import itertools

# Neo4j connection details
uri = "neo4j+s://6e73b041.databases.neo4j.io"
username = "neo4j"
password = "8TH4Gi5XENESIIPJNfTHI6HnM8pRPnEIrpBGSOel8EA"

driver = GraphDatabase.driver(uri, auth=(username, password))

# --- Clear existing data in Neo4j ---
# This is important to ensure a clean state for validation
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("Existing Neo4j data cleared.")

# --- Create Indexes ---
# Ensure indexes are created for the new dynamic labels
unique_labels = set(triplet_related_to_df['subject_type']).union(set(triplet_related_to_df['object_type']))

print(f"Unique entity labels identified for indexing: {unique_labels}")

with driver.session() as session:
    for label in unique_labels:
        clean_label = ''.join(e for e in label if e.isalnum())
        if clean_label:
            query = f"CREATE INDEX IF NOT EXISTS FOR (n:{clean_label}) ON (n.name);"
            session.run(query)
            print(f"Index created for :{clean_label}(name)")
print("All requested indexes have been created/ensured in Neo4j.")

# --- Execute Batch Insertion with dynamic labels ---
batch_size = 1000

print("\nStarting batch insertion with dynamic labels into Neo4j...")

# Group triplets by subject_type and object_type for efficient insertion
# This is necessary because the MERGE query now embeds the label directly
for (sub_type, obj_type), group_df in triplet_related_to_df.groupby(['subject_type', 'object_type']):
    all_triplets_data_grouped = group_df.to_dict(orient='records')
    print(f"Inserting {len(all_triplets_data_grouped)} triplets for Subject Type: {sub_type}, Object Type: {obj_type}")

    with driver.session() as session:
        for i in tqdm(range(0, len(all_triplets_data_grouped), batch_size), desc=f"Inserting {sub_type}-{obj_type} triplets"):
            batch = all_triplets_data_grouped[i : i + batch_size]
            session.execute_write(batch_insert_triplets, batch, sub_type, obj_type)

print("Batch Insertion of RELATED_TO triplets with dynamic labels Completed.")

# Close the driver connection
driver.close()
print("\nNeo4j driver closed.")

Existing Neo4j data cleared.
Unique entity labels identified for indexing: {'DATE', 'ORG', 'PERSON', 'LOCATION'}
Index created for :DATE(name)
Index created for :ORG(name)
Index created for :PERSON(name)
Index created for :LOCATION(name)
All requested indexes have been created/ensured in Neo4j.

Starting batch insertion with dynamic labels into Neo4j...
Inserting 1549 triplets for Subject Type: DATE, Object Type: DATE


Inserting DATE-DATE triplets: 100%|██████████| 2/2 [00:01<00:00,  1.61it/s]


Inserting 395 triplets for Subject Type: LOCATION, Object Type: DATE


Inserting LOCATION-DATE triplets: 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]


Inserting 33 triplets for Subject Type: LOCATION, Object Type: LOCATION


...

Inserting 452 triplets for Subject Type: PERSON, Object Type: ORG


Inserting PERSON-ORG triplets: 100%|██████████| 1/1 [00:00<00:00,  3.43it/s]


Inserting 3554 triplets for Subject Type: PERSON, Object Type: PERSON


Inserting PERSON-PERSON triplets: 100%|██████████| 4/4 [00:01<00:00,  3.19it/s]

Batch Insertion of RELATED_TO triplets with dynamic labels Completed.

Neo4j driver closed.


Neo4j Data Clearing, Index Creation, and Batch Insertion

In [ ]:
import sys
from neo4j import GraphDatabase

def run_validation_query(tx, query_name, query):
    result = tx.run(query).data()
    print(f"\n--- {query_name} ---")
    if result:
        for record in result:
            print(record)
    else:
        print("No results.")

# Re-initialize the Neo4j driver connection
uri = "neo4j+s://6e73b041.databases.neo4j.io"
username = "neo4j"
password = "8TH4Gi5XENESIIPJNfTHI6HnM8pRPnEIrpBGSOel8EA" # Replace with your actual password

# Close driver if it exists and is open, to ensure a clean re-initialization
if 'driver' in globals() and driver is not None:
    try:
        driver.close()
    except Exception as e:
        print(f"Warning: Could not close previous driver instance: {e}", file=sys.stderr)

driver = GraphDatabase.driver(uri, auth=(username, password))

try:
    with driver.session() as session:
        # a. Query to find any nodes with '@' symbol in their name
        run_validation_query(session, "Nodes with '@' in name (PERSON/ORG)",
                             "MATCH (n) WHERE (n:PERSON OR n:ORG) AND n.name CONTAINS '@' RETURN n.name AS email_in_entity")

        # b. Query to find any PERSON or ORG nodes with name length <= 2
        run_validation_query(session, "Short PERSON/ORG entities (length <= 2)",
                             "MATCH (n) WHERE (n:PERSON OR n:ORG) AND size(n.name) <= 2 RETURN n.name AS short_entity, labels(n) AS entity_label")

        # c. Qualitative check for canonicalization effectiveness
        # This requires manual inspection or known variations. For a general check,
        # we can list a few common person names and check for their presence.
        # This assumes canonicalization has merged variations into one form.
        run_validation_query(session, "Canonicalized Names Example (e.g., 'phillip allen')",
                             "MATCH (n:PERSON) WHERE n.name = 'phillip allen' RETURN n.name AS canonical_person_name")

        # If there are specific variations that were expected to be merged, query for them:
        # run_validation_query(session, "Check for specific variations (e.g., 'p allen' should not exist if merged)",
        #                      "MATCH (n:PERSON) WHERE n.name = 'p allen' RETURN n.name AS uncanonicalized_variation")

        # d. Re-verify that there are no isolated nodes in the graph
        run_validation_query(session, "Isolated Nodes (should be empty)",
                             "MATCH (n) WHERE NOT (n)--() RETURN n.name AS isolated_node")

        # e. Re-confirm that nodes possess only their intended specific type label and not the generic 'Entity' label.
        run_validation_query(session, "Nodes with generic ':Entity' label (should be empty after dynamic labeling)",
                             "MATCH (n:Entity) RETURN n.name AS generic_entity_node")

        # f. Query to display the total count of nodes and relationships in the database.
        node_count_query = "MATCH (n) RETURN count(n) AS node_count"
        rel_count_query = "MATCH ()-[r]->() RETURN count(r) AS rel_count"
        total_nodes = session.run(node_count_query).single()['node_count']
        total_rels = session.run(rel_count_query).single()['rel_count']
        print(f"\nTotal nodes in Neo4j after validation: {total_nodes}")
        print(f"Total relationships in Neo4j after validation: {total_rels}")

except Exception as e:
    print(f"Error connecting to Neo4j or executing query: {e}", file=sys.stderr)
finally:
    # It's good practice to close the driver when done with this specific operation
    if driver:
        driver.close()



--- Nodes with '@' in name (PERSON/ORG) ---
No results.

--- Short PERSON/ORG entities (length <= 2) ---
No results.

--- Canonicalized Names Example (e.g., 'phillip allen') ---
{'canonical_person_name': 'phillip allen'}

--- Isolated Nodes (should be empty) ---
No results.

--- Nodes with generic ':Entity' label (should be empty after dynamic labeling) ---
No results.

Total nodes in Neo4j after validation: 676
Total relationships in Neo4j after validation: 7110


# Task
Install the necessary libraries for Milestone 3, including `faiss-cpu`, `sentence-transformers`, and `groq`.

## Install Libraries

### Subtask:
Install all necessary Python libraries for Milestone 3, including `faiss-cpu`, `sentence-transformers`, `groq`, and any others required for the RAG system.


## Summary:

### Data Analysis Key Findings
*   The initial phase of the task involved preparing the development environment by outlining the need to install crucial Python libraries for Milestone 3.
*   The specific libraries identified for installation include `faiss-cpu`, `sentence-transformers`, and `groq`, which are fundamental components for the RAG system.

### Insights or Next Steps
*   The immediate next step is to proceed with the actual installation of the specified libraries to establish the necessary technical foundation for Milestone 3.
